### Dependencies

In [ ]:
!pip install -q --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 137.5 MB/s eta 0:00:00


In [ ]:
!pip install -q rouge-score nltk bert-score requests beautifulsoup4 PyMuPDF

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 74.2 MB/s eta 0:00:00


In [ ]:
!pip show transformers

Name: transformers
Version: 5.5.4
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: huggingface-hub, numpy, packaging, pyyaml, regex, safetensors, tokenizers, tqdm, typer
Required-by: bert-score, peft, sentence-transformers


### Data Collecting (Web Scraping)

#### LOINC

In [ ]:
SECTION7_LOINC = "34073-7"
SECTION12_LOINC = "34090-1"

#### Text Retrieval Functions

In [ ]:
import re
import requests
from bs4 import BeautifulSoup
from xml.etree import ElementTree
import fitz

def extract_setId(url):
  regMatch = re.search(r'[?&]setid=([a-f0-9]{8}-(?:[a-f0-9]{4}-){3}[a-f0-9]{12})', url)

  if regMatch:
    setId = regMatch.group(1)
    print(f"Extracted ID: {setId}")
    return setId
  return None

def getDailyMedDrugInfo(setId):
  try:
    # API endpoint
    endpoint = f"https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/{setId}.xml"

    response = requests.get(endpoint)
    response.raise_for_status()

    tree = ElementTree.fromstring(response.content)

    section7 = tree.find(".//{urn:hl7-org:v3}code[@code='" f"{SECTION7_LOINC}']/..")
    section12 = tree.find(".//{urn:hl7-org:v3}code[@code='" f"{SECTION12_LOINC}']/..")

    section7Text = None
    section12Text = None

    if section7 is not None:
      section7Text = "".join(section7.itertext()).strip()
    if section12 is not None:
      section12Text = "".join(section12.itertext()).strip()

    return section7Text, section12Text
  except Exception as e:
    return getPdfDrugInfo("https://dailymed.nlm.nih.gov/dailymed/downloadpdffile.cfm?setId="+setId)

def getHtmlDrugInfo(url):
  r = requests.get(url.strip())
  r.raise_for_status()

  soup = BeautifulSoup(r.text, 'html.parser')

  section7 = soup.find('div', attrs={"data-sectionCode": SECTION7_LOINC})
  section12 = soup.find('div', attrs={"data-sectionCode": SECTION12_LOINC})

  section7Text = None
  section12Text = None

  if section7 is not None:
    section7Text = section7.text
  if section12 is not None:
    section12Text = section12.text

  if section7Text is None or section12Text is None:
    return getHtmlFallbackDrugInfo(url)

  return section7Text, section12Text

def getHtmlFallbackDrugInfo(url):
    r = requests.get(url.strip())
    r.raise_for_status()
    soup = BeautifulSoup(r.text, 'html.parser')

    for script_or_style in soup(["script", "style"]):
        script_or_style.decompose()

    full_text = soup.get_text(separator="\n")

    # We remove the \n constraint slightly for HTML but keep the 'longest match' logic
    patterns = {
        "7": r"7\s+DRUG\s+INTERACTIONS(.*?)(?=\n\s*\d+\s+[A-Z]|\Z)",
        "12": r"12\s+CLINICAL\s+PHARMACOLOGY(.*?)(?=\n\s*\d+\s+[A-Z]|\Z)"
    }

    results = {"7": None, "12": None}

    for code, pattern in patterns.items():
        matches = re.findall(pattern, full_text, re.IGNORECASE | re.DOTALL)
        if matches:
            # TOC entries are usually ~10-100 chars. Real sections are > 200.
            # max(key=len) effectively skips the TOC entry.
            best_match = max(matches, key=len).strip()
            if len(best_match) > 80:
                results[code] = best_match

    return results["7"], results["12"]

def getPdfDrugInfo(url):
    response = requests.get(url)
    # Using a list to track all matches so we can pick the one that isn't TOC
    found_sections = {"7": None, "12": None}

    with fitz.open(stream=response.content, filetype="pdf") as doc:
        full_text = ""
        for page in doc:
            full_text += page.get_text("text")

    patterns = {
        # This lookahead ensures that 'DRUG INTERACTIONS' is NOT followed by dots/numbers then a newline (TOC style)
        "7": r"(?:\n7\s+DRUG\s+INTERACTIONS\s*\n)(.*?)(?=\n\s*\d+\s+[A-Z]|\Z)",
        "12": r"(?:\n12\s+CLINICAL\s+PHARMACOLOGY\s*\n)(.*?)(?=\n\s*\d+\s+[A-Z]|\Z)"
    }

    for code, pattern in patterns.items():
        # Using findall because the TOC match usually comes first
        # If the specific header-style regex fails, we find all and take the longest one
        matches = re.findall(pattern, full_text, re.IGNORECASE | re.DOTALL)
        if matches:
            # Sort by length and take the longest one (actual content > TOC entry)
            found_sections[code] = max(matches, key=len).strip()
        else:
            found_sections[code] = None

    return found_sections["7"], found_sections["12"]

#### Load URLs from Excel

In [ ]:
import pandas as pd
import io
from google.colab import files

print("Click the button below to upload your Excel file...")
uploaded = files.upload()

df = pd.read_excel(io.BytesIO(list(uploaded.values())[0]))
urls = df['URL'].tolist()
print(f"\nLoaded {len(urls)} URLs successfully\n")

Click the button below to upload your Excel file...


Saving 100 Drug URLs 2026_04_07.xlsx to 100 Drug URLs 2026_04_07.xlsx

Loaded 100 URLs successfully



In [ ]:
import requests

def getDrugText(url):
    try:
        response_head = requests.head(url, allow_redirects=True, timeout=10)
        content_type = response_head.headers.get('Content-Type', '').lower()

        if 'application/pdf' in content_type or url.strip().lower().endswith(".pdf"):
            print(f"Detected PDF")
            return getPdfDrugInfo(url)
        elif 'text/html' in content_type:
            print(f"Detected HTML")
            return getHtmlDrugInfo(url)
        else:
            print(f"[!] Unknown content type: {content_type}")

    except Exception as e:
        print(f"[!] Error accessing URL: {e}")

In [ ]:
import io
import time

master_drug_data = []
failures = []

def extract_drug_text(url):
  try:
    setId = extract_setId(url)
    if setId is not None:
      section7Text, section12Text = getDailyMedDrugInfo(setId)
      if section7Text is None or section12Text is None:
        section7Text, section12Text = getDrugText(url)
    else:
      section7Text, section12Text = getDrugText(url)

    if section7Text is None and section12Text is None:
      return "NA", "NA", False

    return section7Text, section12Text, True
  except Exception as e:
    return "NA", str(e), False

print("Scraping URLs...\n")

def add_err(iter, err):
  failures.append({"index": iter+1, "url": url, "error": err})
  print(f"  [{iter+1}/{len(urls)}] FAILED — {url[:70]}, failure: {len(failures)}, error: {err}")

for i, url in enumerate(urls):
  print(f"Processing {url}")

  try:
    sec7, sec12, success = extract_drug_text(url)
    if success and (
      (sec7 is not None and len(sec7) > 80) or
      (sec12 is not None and len(sec12) > 80)
    ):
      master_drug_data.append({"index": i+1, "url": url, "section7": sec7, "section12": sec12})
      print(f"  [{i+1}/{len(urls)}] OK — {url[:70]}")
    else:
      add_err(i, sec12)
  except Exception as e:
    add_err(i, str(e))

  print("----------------------------------------")
  time.sleep(0.15)

print(f"\n✓ Done — {len(master_drug_data)} succeeded, {len(failures)} failed.")

Scraping URLs...

Processing https://labeling.pfizer.com/ShowLabeling.aspx?id=16474 
Detected PDF
  [1/100] OK — https://labeling.pfizer.com/ShowLabeling.aspx?id=16474 
----------------------------------------
Processing https://dailymed.nlm.nih.gov/dailymed/fda/fdaDrugXsl.cfm?setid=ca73b519-015a-436d-aa3c-af53492825a1&type=display
Extracted ID: ca73b519-015a-436d-aa3c-af53492825a1
  [2/100] OK — https://dailymed.nlm.nih.gov/dailymed/fda/fdaDrugXsl.cfm?setid=ca73b51
----------------------------------------
Processing https://uspl.lilly.com/verzenio/verzenio.html#pi
Detected HTML
  [3/100] OK — https://uspl.lilly.com/verzenio/verzenio.html#pi
----------------------------------------
Processing https://www.janssenlabels.com/package-insert/product-monograph/prescribing-information/ZYTIGA-pi.pdf
Detected PDF
  [4/100] OK — https://www.janssenlabels.com/package-insert/product-monograph/prescri
----------------------------------------
Processing https://dailymed.nlm.nih.gov/dailymed/fda/fdaD

#### Save Collected Data to CSV

In [ ]:
import csv

keys = master_drug_data[0].keys()

with open('collected_drug_data.csv', 'w', newline='') as output_file:
    dict_writer = csv.DictWriter(output_file, keys)
    dict_writer.writeheader()
    dict_writer.writerows(master_drug_data)

print("Data saved to 'collected_drug_data.csv'")

Data saved to 'collected_drug_data.csv'


In [ ]:
import csv

keys = failures[0].keys()

with open('failures_drug_data.csv', 'w', newline='') as output_file:
    dict_writer = csv.DictWriter(output_file, keys)
    dict_writer.writeheader()
    dict_writer.writerows(failures)

print("Failures saved to 'failures_drug_data.csv'")

Failures saved to 'failures_drug_data.csv'


### Check Largest Text

In [ ]:
def find_best_entry(data_list):
    """
    Returns the complete dictionary that contains the longest
    text in either section 7 or section 12.
    """

    def scoring_criteria(d):
        # We handle None or missing keys by defaulting to an empty string
        s7_content = d.get('section7') or ""
        s12_content = d.get('section12') or ""

        # The 'score' is the length of the longest string found in this dict
        return max(len(s7_content), len(s12_content))

    if not data_list:
        print("Debug: List is empty.")
        return None

    # max() returns the FULL dictionary 'd' from the list
    best_dict = max(data_list, key=scoring_criteria)

    # --- Debug Summary ---
    print(f"--- Winner Stats ---")
    print(f"Section 7 Length:  {len(best_dict.get('section7') or '')}")
    print(f"Section 12 Length: {len(best_dict.get('section12') or '')}")
    print(f"Full Dict Keys:    {list(best_dict.keys())}")

    return best_dict

result = find_best_entry(master_drug_data)
print(f"""
Final Result URL: {result['url']}
 Section7 Length: {len(result['section7'])}
Section12 Length: {len(result['section12'])}
    Section7 txt: {result['section7'][:250]}
   Section12 txt: {result['section12'][:250]}
""")

--- Winner Stats ---
Section 7 Length:  13291
Section 12 Length: 32709
Full Dict Keys:    ['index', 'url', 'section7', 'section12']

Final Result URL: https://dailymed.nlm.nih.gov/dailymed/fda/fdaDrugXsl.cfm?setid=696f9e80-9cae-403b-de9e-078343ce4713&type=display
 Section7 Length: 13291
Section12 Length: 32709
    Section7 txt: 7	DRUG INTERACTIONS
               
               
                  
                     
                        See Full Prescribing Information for a list of clinically significant drug interactions. (4, 5.1, 5.2, 5.3, 7.1, 7.2)
               
   Section12 txt: 12	CLINICAL PHARMACOLOGY
               
               
                  
                     
                     
                     12.1	Mechanism of Action
                     
                        Aprepitant is a selective high-affinit



### Data Injecting

#### Optionally Provide a CSV Drug Data File

In [ ]:
from google.colab import files
import csv
import sys

print("Click the button below to upload the CSV file...")
uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

filename = next(iter(uploaded))

csv.field_size_limit(sys.maxsize)

master_drug_data = []
with open(filename, newline='') as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        master_drug_data.append(row)

print(master_drug_data)

Click the button below to upload the CSV file...


Saving drug_data_LongT5_results.csv to drug_data_LongT5_results.csv
User uploaded file "drug_data_LongT5_results.csv" with length 793781 bytes
[{'index': '1', 'url': 'https://labeling.pfizer.com/ShowLabeling.aspx?id=16474 ', 'section7': '7.1 \nPotential for PAXLOVID to Affect Other Drugs \n \nPAXLOVID (nirmatrelvir co-packaged with ritonavir) is a strong inhibitor of CYP3A, and an inhibitor of \nCYP2D6, P-gp and OATP1B1. Co-administration of PAXLOVID with drugs that are primarily \nmetabolized by CYP3A and CYP2D6 or are transported by P-gp or OATP1B1 may result in increased \nplasma concentrations of such drugs and increase the risk of adverse events. Co-administration of \nPAXLOVID with drugs highly dependent on CYP3A for clearance and for which elevated plasma \nconcentrations are associated with serious and/or life-threatening events is contraindicated [see \nContraindications (4) and Drug Interactions (7.3) Table 2]. Co-administration with other CYP3A \nsubstrates may require a dos

In [ ]:
completed = []
for model, col in [("LongT5", "LongT5_Summary_0"), ("MedGemma", "MedGemma_Summary_0"), ("Gemma4", "Gemma4_Summary_0")]:
    if col in master_drug_data[0]:
        completed.append(model)

print(f"\n✓ Detected completed model runs: {completed or 'None'}")
print(f"  → You may SKIP the inference blocks for: {completed}")


✓ Detected completed model runs: ['LongT5']
  → You may SKIP the inference blocks for: ['LongT5']


#### Analyze Data Length

In [ ]:
from transformers import AutoTokenizer

diagnostic_tokenizer = AutoTokenizer.from_pretrained("google/long-t5-tglobal-base")

for label, section in [("section7", "Drug Interactions"), ("section12", "Clinical Pharmacology")]:
    lengths = [
        len(diagnostic_tokenizer(d[label])["input_ids"])
        for d in master_drug_data
        if d.get(label) is not None and len(d[label]) > 80
    ]
    if lengths:
        print(f"{section}: min={min(lengths)}, max={max(lengths)}, avg={round(sum(lengths)/len(lengths))}")

config.json:   0%|          | 0.00/851 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Drug Interactions: min=17, max=4994, avg=700
Clinical Pharmacology: min=115, max=4307, avg=1300


### Summary Instructions

In [ ]:
INSTRUCTIONS_SEC7 = (
    "### TASK: MEDICAL SUMMARY — DRUG INTERACTIONS ###\n"
    "You are a clinical pharmacist. Summarize the drug interactions section below "
    "for a prescribing clinician. Follow these rules strictly:\n"
    "1. DO NOT copy sentences verbatim. Rewrite in concise, professional clinical language.\n"
    "2. COVER these domains if present: interacting drugs or drug classes, "
    "mechanism of interaction, clinical consequence, and any recommended management.\n"
    "3. If a domain is absent from the source, omit it rather than stating it is missing.\n"
    "4. Respond with the summary only. No preamble, no commentary, no closing remarks.\n"
    "5. If the source contains substantial detail, aim for 150-400 characters. "
    "If the source is sparse, a shorter response is acceptable. Never exceed 400 characters.\n\n"
    "--- SOURCE TEXT ---\n"
)

INSTRUCTIONS_SEC12 = (
    "### TASK: MEDICAL SUMMARY — CLINICAL PHARMACOLOGY ###\n"
    "You are a clinical pharmacist. Summarize the clinical pharmacology section below "
    "for a prescribing clinician. Follow these rules strictly:\n"
    "1. DO NOT copy sentences verbatim. Rewrite in concise, professional clinical language.\n"
    "2. COVER these domains if present: mechanism of action, pharmacodynamics, "
    "absorption, distribution, metabolism, and elimination.\n"
    "3. If a domain is absent from the source, omit it rather than stating it is missing.\n"
    "4. Respond with the summary only. No preamble, no commentary, no closing remarks.\n"
    "5. If the source contains substantial detail, aim for 150-400 characters. "
    "If the source is sparse, a shorter response is acceptable. Never exceed 400 characters.\n"
    "--- SOURCE TEXT ---\n"
)

### Global Model Variables

In [ ]:
MAX_NEW_TOKENS = 500

### LongT5 Inference

#### Remove MedGemma From Memory

In [ ]:
import gc
import torch

try:
    del model_mg
    del pipe_mg
    gc.collect()
    torch.cuda.empty_cache()
    print(f"GPU cleared. Free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")
except:
    print("No MedGemma in memory, continuing...")

No MedGemma in memory, continuing...


#### Iterate Over Drug Data

In [ ]:
import torch
import time
import warnings
import textwrap
from transformers import AutoTokenizer, LongT5ForConditionalGeneration

tokenizer_lt5 = None
model_lt5 = None

def load_long_t5():
  global tokenizer_lt5, model_lt5
  print("\nLoading Long-T5...")
  tokenizer_lt5 = AutoTokenizer.from_pretrained("google/long-t5-tglobal-base")
  model_lt5 = LongT5ForConditionalGeneration.from_pretrained(
      "google/long-t5-tglobal-base",
      torch_dtype=torch.float16
  ).to("cuda")
  print("Long-T5 loaded!\n")

def summarize_long_t5(text):
    prompt = text
    inputs = tokenizer_lt5(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=16384
    ).to(model_lt5.device)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        output_ids = model_lt5.generate(
            inputs["input_ids"],
            min_new_tokens=80,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=4,
            no_repeat_ngram_size=3,
            length_penalty=1.0,
            early_stopping=True,
        )
    return tokenizer_lt5.decode(output_ids[0], skip_special_tokens=True)

print("Running Long-T5 on all drugs (this will take a while, do not interrupt)...")
lt5_results = []
lt5_times   = []

num_texts = len(master_drug_data) * 2

if master_drug_data and f"LongT5_Summary_0" in master_drug_data[0]:
    print("LongT5 results already present in loaded data. Skipping inference.")

    lt5_results = [d[f"LongT5_Summary_{j}"] for d in master_drug_data for j in range(2) if d.get(f"LongT5_Summary_{j}")]
    lt5_times   = [float(d[f"LongT5_Time_{j}"]) for d in master_drug_data for j in range(2) if d.get(f"LongT5_Time_{j}")]
else:
    if tokenizer_lt5 is None or model_lt5 is None:
        load_long_t5()
    processed = 0
    for i, data in enumerate(master_drug_data):
        for j, text in enumerate([data["section7"], data["section12"]]):
            if text is None or len(text) < 80:
                continue
            start = time.time()
            output = summarize_long_t5(text)
            elapsed = round(time.time() - start, 2)
            lt5_results.append(output)
            lt5_times.append(elapsed)
            master_drug_data[i][f"LongT5_Summary_{j}"] = output
            master_drug_data[i][f"LongT5_Time_{j}"] = elapsed

            processed += 1

            if processed % 5 == 0 or processed == num_texts:
                free_gb = torch.cuda.mem_get_info()[0]/1e9
                print(f"  Progress: {processed}/{num_texts} done...GPU free: {free_gb:.2f} GB")

lt5_avg_time = round(sum(lt5_times)/len(lt5_times),2)

print(f"\n✓ LongT5 done. Avg time: {lt5_avg_time}s")

Running Long-T5 on all drugs (this will take a while, do not interrupt)...
LongT5 results already present in loaded data. Skipping inference.

✓ LongT5 done. Avg time: 4.33s


#### Save Results to CSV

In [ ]:
import csv

keys = master_drug_data[0].keys()

with open('drug_data_LongT5_results.csv', 'w', newline='') as output_file:
    dict_writer = csv.DictWriter(output_file, keys)
    dict_writer.writeheader()
    dict_writer.writerows(master_drug_data)

print("Data saved to 'drug_data_LongT5_results.csv'")

Data saved to 'drug_data_LongT5_results.csv'


### MedGemma Inference

#### Remove LongT5 From Memory

In [ ]:
import os
import gc
import torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

try:
    del model_lt5
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print(f"GPU cleared. Free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

GPU cleared. Free: 41.96 GB


#### Load MedGemma

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

tokenizer_mg = None
model_mg = None

def load_medgemma():
    global tokenizer_mg, model_mg, pipe_mg
    print("\nLoading MedGemma...")
    tokenizer_mg = AutoTokenizer.from_pretrained("google/medgemma-4b-it")
    model_mg = AutoModelForCausalLM.from_pretrained(
        "google/medgemma-4b-it",
        dtype=torch.bfloat16,
        device_map="auto"
    )
    pipe_mg = pipeline(
        "text-generation",
        model=model_mg,
        tokenizer=tokenizer_mg
    )
    print("MedGemma loaded!\n")

#### Iterate Over Drug Data

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import gc
import torch
import time
import warnings
import textwrap

def medgemma_summarize_text(text, instructions):
    prompt = instructions + text
    messages  = [{"role": "user", "content": prompt}]
    formatted = tokenizer_mg.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        output = pipe_mg(
            formatted,
            return_full_text=False,
            min_new_tokens=80,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=4,
            no_repeat_ngram_size=3,
            length_penalty=1.0,
            early_stopping=True,
            do_sample=False,
        )
    return output[0]["generated_text"].strip()

print("Running MedGemma on all drugs (do not interrupt)...")
mg_results = []
mg_times   = []

num_texts = len(master_drug_data) * 2

if master_drug_data and f"MedGemma_Summary_0" in master_drug_data[0]:
    print("MedGemma results already present in loaded data. Skipping inference.")

    mg_results = [d[f"MedGemma_Summary_{j}"] for d in master_drug_data for j in range(2) if d.get(f"MedGemma_Summary_{j}")]
    mg_times   = [float(d[f"MedGemma_Time_{j}"]) for d in master_drug_data for j in range(2) if d.get(f"MedGemma_Time_{j}")]
else:
    if tokenizer_mg is None or model_mg is None:
        load_medgemma()
    processed = 0
    for i, data in enumerate(master_drug_data):
        if i > 0 and i % 5 == 0:
            gc.collect()
            torch.cuda.empty_cache()

        for j, text in enumerate([data["section7"], data["section12"]]):
            if text is None or len(text) < 80:
              continue
            start = time.time()
            instructions = INSTRUCTIONS_SEC7 if j == 0 else INSTRUCTIONS_SEC12
            output = medgemma_summarize_text(text, instructions)
            elapsed = round(time.time() - start, 2)
            mg_results.append(output)
            mg_times.append(elapsed)
            master_drug_data[i][f"MedGemma_Summary_{j}"] = output
            master_drug_data[i][f"MedGemma_Time_{j}"] = elapsed

            processed += 1

            if processed % 5 == 0 or processed == num_texts:
                free_gb = torch.cuda.mem_get_info()[0]/1e9
                print(f"  Progress: {processed}/{num_texts} done...GPU free: {free_gb:.2f} GB")

mg_avg_time = round(sum(mg_times)/len(mg_times),2)

print(f"\n✓ MedGemma done. Avg time: {mg_avg_time}s")

Running MedGemma on all drugs (do not interrupt)...

Loading MedGemma...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

  Progress: 130/144 done...GPU free: 28.46 GB


Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=80) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=80) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

  Progress: 135/144 done...GPU free: 28.46 GB


Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=80) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=80) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma


✓ MedGemma done. Avg time: 17.13s


#### Save Results to CSV

In [ ]:
import csv

keys = master_drug_data[0].keys()

with open('drug_data_MedGemma_results.csv', 'w', newline='') as output_file:
    dict_writer = csv.DictWriter(output_file, keys)
    dict_writer.writeheader()
    dict_writer.writerows(master_drug_data)

print("Data saved to 'drug_data_MedGemma_results.csv'")

Data saved to 'drug_data_MedGemma_results.csv'


### Gemma 4 Inference

#### Remove MedGemma From Memory

In [ ]:
import gc
import torch

try:
    del model_mg
    del pipe_mg
    gc.collect()
    torch.cuda.empty_cache()
    print(f"GPU cleared. Free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")
except:
    print("No MedGemma in memory, continuing...")

GPU cleared. Free: 41.84 GB


#### Load Gemma 4

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

MODEL_ID = "google/gemma-4-e4b-it"

g4_tokenizer = None
g4_model = None

def load_gemma4():
  global g4_tokenizer, g4_model
  print(f"Loading model '{MODEL_ID}'...")
  g4_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
  g4_model = AutoModelForCausalLM.from_pretrained(
      MODEL_ID,
      device_map="auto",
      dtype=torch.bfloat16,
  )
  print("Model loaded successfully.\n")

#### Gemma 4 Summarize Function

In [ ]:
def gemma4_summarize_text(text, instructions):
    if g4_tokenizer is None or g4_model is None:
        load_gemma4()
    messages = [
        {
            "role": "user",
            "content": instructions + text,
        }
    ]

    # Apply the chat template and tokenize
    inputs = g4_tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        return_dict=True,
        add_generation_prompt=True,
    ).to(g4_model.device)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        outputs = g4_model.generate(
            **inputs,
            min_new_tokens=80,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=4,
            no_repeat_ngram_size=3,
            length_penalty=1.0,
            early_stopping=True,
        )

    summary = g4_tokenizer.decode(
        outputs[0][input_len:],
        skip_special_tokens=True,
    ).strip()

    return summary

#### Test Summary

In [ ]:
gemma4_summarize_text(
"""
7.1 Effect of Other Drugs on EPIDIOLEX
Strong CYP3A4 or CYP2C19 Inducers
Concomitant use with a strong CYP3A4 and CYP2C19 inducer (rifampin 600 mg once daily) decreased
cannabidiol and 7-OH-CBD plasma concentrations by approximately 32% and 63%. The impact of such
changes on efficacy of EPIDIOLEX is not known [see Clinical Pharmacology (12.3)]. Consider an increase in
EPIDIOLEX dosage (based on clinical response and tolerability) up to 2-fold, when concomitantly used with a
strong CYP3A4 and/or CYP2C19 inducer.
7.2 Effect of EPIDIOLEX on Other Drugs
Antiepileptic Drugs
Clobazam
Concomitant use of EPIDIOLEX with clobazam increases plasma concentrations of N-desmethylclobazam, the
active metabolite of clobazam [see Clinical Pharmacology (12.3)], which may increase the risk of clobazam-
related adverse reactions [see Adverse Reactions (6.1) and Warnings and Precautions (5.1, 5.2)]. Consider a
reduction in dosage of clobazam if adverse reactions known to occur with clobazam are experienced when
concomitantly used with EPIDIOLEX.
Stiripentol
Concomitant use of EPIDIOLEX with stiripentol increases plasma exposures of stiripentol [see Clinical
Pharmacology (12.3)]. Monitor for stiripentol-related adverse reactions when concomitantly used with
EPIDIOLEX.
Orally Administered P-gp Substrates
Concomitant use of EPIDIOLEX with orally administered everolimus results in an approximately 2.5-fold
increase in plasma exposures of everolimus [see Clinical Pharmacology (12.3)]. When initiating EPIDIOLEX
in patients taking everolimus, monitor therapeutic drug levels of everolimus and adjust the dosage accordingly.
In patients on a stable dosage of EPIDIOLEX, it is recommended to initiate everolimus at a lower starting
dosage and titrate the dose based on therapeutic drug monitoring.
Increases in exposure of other orally administered P-gp substrates (e.g., sirolimus, tacrolimus, digoxin) may be
observed when concomitantly used with EPIDIOLEX. Consider therapeutic drug monitoring and dosage
reduction of other P-gp substrates when given orally with EPIDIOLEX.
CYP1A2, CYP2B6, CYP2C8, CYP2C19, and UGT1A9 Substrates
CYP1A2 Substrates
Cannabidiol is a weak inhibitor of CYP1A2 [see Clinical Pharmacology (12.3)]. Increases in exposure of
certain CYP1A2 substrates (e.g., theophylline, tizanidine) may be observed when concomitantly used with
EPIDIOLEX. Consider dosage reduction of CYP1A2 substrates where minimal concentration changes may lead
to serious adverse reactions, as clinically appropriate, when concomitantly used with EPIDIOLEX.
CYP2B6 Substrates
Cannabidiol is an inducer and inhibitor of CYP2B6 [see Clinical Pharmacology (12.3)]. No clinically
significant reduction in exposures of CYP2B6 substrates are observed when concomitantly used with
EPIDIOLEX at 7.5 mg/kg twice daily. Changes in exposures of CYP2B6 substrates are unknown when
concomitantly used with EPIDIOLEX at doses above 7.5 mg/kg twice daily. Consider dosage modification of
CYP2B6 substrates, as clinically appropriate, when concomitantly used with EPIDIOLEX at doses above
7.5 mg/kg twice daily.
CYP2C8 Substrates
Concomitant use of EPIDIOLEX may cause clinically significant interactions with CYP2C8 substrates.
Consider a reduction in dosage of CYP2C8 substrates, as clinically appropriate, if adverse reactions are
experienced when concomitantly used with EPIDIOLEX.
CYP2C19 Substrates
Cannabidiol is a moderate inhibitor of CYP2C19 [see Clinical Pharmacology (12.3)]. Concomitant use of
EPIDIOLEX increases plasma concentrations of CYP2C19 substrates and may increase the risk of adverse
reactions. Consider a dosage reduction of CYP2C19 substrates, as clinically appropriate, when concomitantly
used with EPIDIOLEX.
For CYP2C19 substrates (e.g., clopidogrel) where efficacy is mainly due to their active metabolite(s),
concomitant use of EPIDIOLEX may decrease plasma concentration of the active metabolite(s) and may
therefore decrease efficacy. Consider a dosage increase of such CYP2C19 substrates, as clinically appropriate,
when concomitantly used with EPIDIOLEX.
UGT1A9 Substrates
Cannabidiol is an inhibitor of UGT1A9 [see Clinical Pharmacology (12.3)]. Increases in exposure of UGT1A9
substrates may be observed when concomitantly used with EPIDIOLEX. Consider a reduction in dosage of
UGT1A9 substrates where minimal concentration changes may lead to serious adverse reactions, as clinically
appropriate, when concomitantly used with EPIDIOLEX.
7.3 Concomitant Use of EPIDIOLEX and Valproate
Concomitant use of EPIDIOLEX and valproate increases the incidence of liver enzyme elevations [see
Warnings and Precautions (5.1)]. If such elevations occur, consider discontinuation or reduction of
EPIDIOLEX and/or concomitant valproate. Insufficient data are available to assess the risk of concomitant
administration of other hepatotoxic drugs and EPIDIOLEX.
7.4 CNS Depressants and Alcohol
Concomitant use of EPIDIOLEX with other CNS depressants (including alcohol) may increase the risk of
sedation and somnolence [see Warnings and Precautions (5.2)].
""", INSTRUCTIONS_SEC7)

Loading model 'google/gemma-4-e4b-it'...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Model loaded successfully.



'Strong inducers decrease CBD concentrations; consider up to a 2x dose increase. With Clobazam, increased active metabolite may raise adverse reaction risk; reduce dose if needed. EPIDDIOLEX increases exposures of many substrates (P-gp, CYP, UGT); dose adjustments or monitoring are advised. Co-administration with Valproic acid increases liver enzyme risk. CNS depressant use heightens sedation risk.'

#### Iterate Over Drug Data

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import gc
import torch
import time
import warnings
import textwrap

print("Running Gemma 4 on all drugs (do not interrupt)...")
g4_results = []
g4_times   = []

num_texts = len(master_drug_data) * 2

if master_drug_data and f"Gemma4_Summary_0" in master_drug_data[0]:
    print("Gemma4 results already present in loaded data. Skipping inference.")

    g4_results = [d[f"Gemma4_Summary_{j}"] for d in master_drug_data for j in range(2) if d.get(f"Gemma4_Summary_{j}")]
    g4_times   = [float(d[f"Gemma4_Time_{j}"]) for d in master_drug_data for j in range(2) if d.get(f"Gemma4_Time_{j}")]
else:
    if g4_tokenizer is None or g4_model is None:
        load_gemma4()
    processed = 0
    for i, data in enumerate(master_drug_data):
        if i > 0 and i % 5 == 0:
            gc.collect()
            torch.cuda.empty_cache()

        for j, text in enumerate([data["section7"], data["section12"]]):
            if text is None or len(text) < 80:
              continue
            start = time.time()
            instructions = INSTRUCTIONS_SEC7 if j == 0 else INSTRUCTIONS_SEC12
            output = gemma4_summarize_text(text, instructions)
            elapsed = round(time.time() - start, 2)
            g4_results.append(output)
            g4_times.append(elapsed)
            master_drug_data[i][f"Gemma4_Summary_{j}"] = output
            master_drug_data[i][f"Gemma4_Time_{j}"] = elapsed

            processed += 1

            if processed % 5 == 0 or processed == num_texts:
                free_gb = torch.cuda.mem_get_info()[0]/1e9
                print(f"  Progress: {processed}/{num_texts} done...GPU free: {free_gb:.2f} GB")

g4_avg_time = round(sum(g4_times)/len(g4_times),2)

print(f"\n✓ Gemma 4 done. Avg time: {g4_avg_time}s")

Running Gemma 4 on all drugs (do not interrupt)...
  Progress: 5/144 done...GPU free: 17.62 GB
  Progress: 10/144 done...GPU free: 17.57 GB
  Progress: 15/144 done...GPU free: 23.09 GB
  Progress: 20/144 done...GPU free: 25.50 GB
  Progress: 25/144 done...GPU free: 22.76 GB
  Progress: 30/144 done...GPU free: 24.85 GB
  Progress: 35/144 done...GPU free: 22.03 GB
  Progress: 40/144 done...GPU free: 24.20 GB
  Progress: 45/144 done...GPU free: 21.90 GB
  Progress: 50/144 done...GPU free: 21.17 GB
  Progress: 55/144 done...GPU free: 14.51 GB
  Progress: 60/144 done...GPU free: 16.83 GB
  Progress: 65/144 done...GPU free: 16.83 GB
  Progress: 70/144 done...GPU free: 22.21 GB
  Progress: 75/144 done...GPU free: 22.21 GB
  Progress: 80/144 done...GPU free: 24.48 GB
  Progress: 85/144 done...GPU free: 24.05 GB
  Progress: 90/144 done...GPU free: 21.22 GB
  Progress: 95/144 done...GPU free: 21.22 GB
  Progress: 100/144 done...GPU free: 24.59 GB
  Progress: 105/144 done...GPU free: 18.67 GB
  P

#### Save Results to CSV

In [ ]:
import csv

keys = master_drug_data[0].keys()

with open('drug_data_Gemma4_results.csv', 'w', newline='') as output_file:
    dict_writer = csv.DictWriter(output_file, keys)
    dict_writer.writeheader()
    dict_writer.writerows(master_drug_data)

print("Data saved to 'drug_data_Gemma4_results.csv'")

Data saved to 'drug_data_Gemma4_results.csv'


### Results

#### Setup for BERTScores

In [ ]:
import logging
import os

from transformers import AutoTokenizer, AutoModel
from bert_score import BERTScorer
import torch

logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def wrap_output(text, width=80, indent="  "):
    return "\n".join(textwrap.wrap(
        text.strip(), width=width,
        initial_indent=indent, subsequent_indent=indent
    ))

def get_bertscore_batch(hypotheses, references):
    MED_MODEL = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"

    tokenizer = AutoTokenizer.from_pretrained(MED_MODEL)

    if not hasattr(tokenizer, "build_inputs_with_special_tokens"):
        def build_inputs_with_special_tokens(token_ids_0, token_ids_1=None):
            sep = [tokenizer.sep_token_id]
            cls = [tokenizer.cls_token_id]
            if token_ids_1 is None:
                return cls + token_ids_0 + sep
            return cls + token_ids_0 + sep + token_ids_1 + sep
        tokenizer.build_inputs_with_special_tokens = build_inputs_with_special_tokens

    if tokenizer.model_max_length > 1_000_000:
        tokenizer.model_max_length = 512

    scorer = BERTScorer(
        model_type=MED_MODEL,
        num_layers=9,
        batch_size=16,
        lang="en",
        use_fast_tokenizer=True
    )

    scorer._tokenizer = tokenizer

    with torch.no_grad():
        P, R, F1 = scorer.score(hypotheses, references)

    return [round(f.item(), 4) for f in F1]

references = []
for data in master_drug_data:
    for text in [data["section7"], data["section12"]]:
        if text is None or len(text) < 80:
            continue
        references.append(text)

print(len(lt5_results), len(mg_results), len(g4_results), len(references))

139 139 139 139


#### Calculate BERTScores

In [ ]:
print("Scoring all outputs with BERTScore...")
lt5_scores = get_bertscore_batch(lt5_results, references)
mg_scores  = get_bertscore_batch(mg_results,  references)
g4_scores  = get_bertscore_batch(g4_results,  references)
print("Done.\n")

Scoring all outputs with BERTScore...


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Done.



In [ ]:
print(lt5_scores, mg_scores, g4_scores)

[0.8738, 0.8944, 0.8875, 0.825, 0.8584, 0.7754, 0.8664, 0.8889, 0.9209, 0.7894, 0.8932, 0.8647, 0.8264, 0.7755, 0.8985, 0.8866, 0.7874, 0.8649, 0.8641, 0.9103, 0.8079, 0.8674, 0.7852, 0.8546, 0.8913, 0.8765, 0.8585, 0.8603, 0.7963, 0.8723, 0.8168, 0.867, 0.8643, 0.8282, 0.9104, 0.8553, 0.8811, 0.8313, 0.8583, 0.8202, 0.8775, 0.8605, 0.8518, 0.8527, 0.8305, 0.8386, 0.884, 0.8696, 0.8454, 0.7383, 0.8319, 0.8016, 0.8619, 0.849, 0.875, 0.888, 0.8513, 0.8395, 0.8505, 0.7914, 0.8228, 0.8403, 0.8571, 0.868, 0.8681, 0.8587, 0.8637, 0.9211, 0.8627, 0.7782, 0.9476, 0.8766, 0.8756, 0.8454, 0.8407, 0.868, 0.8652, 0.8628, 0.8461, 0.8926, 0.8436, 0.8574, 0.8274, 0.9047, 0.9156, 0.8971, 0.7913, 0.8231, 0.8035, 0.8812, 0.8526, 0.8252, 0.8717, 0.8366, 0.8611, 0.8566, 0.9042, 0.8336, 0.8906, 0.8528, 0.8629, 0.8939, 0.7889, 0.8585, 0.8068, 0.8226, 0.8275, 0.8664, 0.9435, 0.8763, 0.8474, 0.8505, 0.7843, 0.8938, 0.8647, 0.8616, 0.8239, 0.8726, 0.7711, 0.8279, 0.726, 0.878, 0.8144, 0.8832, 0.8339, 0.8883, 0

#### Add BERTScores to Data

In [ ]:
def add_bert_scores(data_list, score_list, prefix=""):
    scored_pairs = [
        (i, j)
        for i, d in enumerate(data_list)
        for j in range(2)
        if d.get(f"section{7 if j == 0 else 12}") and len(str(d.get(f"section{7 if j == 0 else 12}", ""))) >= 80
    ]

    if len(scored_pairs) != len(score_list):
        raise ValueError(
            f"Mismatched lengths: {len(scored_pairs)} scored pairs but got {len(score_list)} scores."
        )

    sec_keys = ["section7", "section12"]
    score_keys = [f"{prefix}BERTScore_Sec7", f"{prefix}BERTScore_Sec12"]

    for (i, j), score in zip(scored_pairs, score_list):
        data_list[i][score_keys[j]] = score

    return data_list

In [ ]:
master_drug_data = add_bert_scores(master_drug_data, lt5_scores, "LongT5_")

In [ ]:
master_drug_data = add_bert_scores(master_drug_data, mg_scores, "MedGemma_")

In [ ]:
master_drug_data = add_bert_scores(master_drug_data, g4_scores, "Gemma4_")

#### Save results to CSV

In [ ]:
import csv

keys = master_drug_data[0].keys()

with open('drug_data_BERTScore_results.csv', 'w', newline='') as output_file:
    dict_writer = csv.DictWriter(output_file, keys)
    dict_writer.writeheader()
    dict_writer.writerows(master_drug_data)

print("Data saved to 'drug_data_BERTScore_results.csv'")

Data saved to 'drug_data_BERTScore_results.csv'


#### Select Three Examples

In [ ]:
import random

showcase_indices = []

for i in range(3):
    random_index = random.randint(0, len(master_drug_data) - 1)
    showcase_indices.append(random_index)

showcase_cases = [master_drug_data[i] for i in showcase_indices if i < len(master_drug_data)]

print(f"Showcase cases  : {len(showcase_cases) * 2}")
print(f"Total test cases: {len(master_drug_data) * 2}")
for c in showcase_cases:
    print(f"  [{c['index']}] {c['url'][:70]}")

Showcase cases  : 6
Total test cases: 144
  [42] https://dailymed.nlm.nih.gov/dailymed/fda/fdaDrugXsl.cfm?setid=fc22e8d
  [93] https://dailymed.nlm.nih.gov/dailymed/fda/fdaDrugXsl.cfm?setid=0cbce38
  [84] https://www.gene.com/download/pdf/zelboraf_prescribing.pdf


#### Showcase Examples

In [ ]:
print("="*70)
print("  SHOWCASE — 3 EXAMPLE SUMMARIES")
print("="*70)

for case in showcase_cases:
    idx = next(i for i, c in enumerate(master_drug_data) if c["url"] == case["url"])

    print(f"\n{'─'*70}")
    print(f"  DRUG {idx+1}")
    print(f"  Source: {case['url'][:70]}")
    print(f"{'─'*70}")

    # Show section 7 if available, fall back to section 12
    original_text = case.get("section7") or case.get("section12") or ""
    print(f"\n  ORIGINAL TEXT (first 150 words):")
    print(wrap_output(" ".join(original_text.split()[:150])))

    lt5_s7_score  = lt5_scores[idx * 2]
    lt5_s12_score = lt5_scores[idx * 2 + 1]

    mg_s7_score   = mg_scores[idx * 2]
    mg_s12_score  = mg_scores[idx * 2 + 1]

    g4_s7_score   = g4_scores[idx * 2]
    g4_s12_score  = g4_scores[idx * 2 + 1]

    print(f"\n  LONG-T5 SUMMARY (Section 7):")
    print(wrap_output(case.get("LongT5_Summary_0") or "N/A"))
    print(f"  LONG-T5 SUMMARY (Section 12):")
    print(wrap_output(case.get("LongT5_Summary_1") or "N/A"))
    print(f"\n  Time (s7): {case.get('LongT5_Time_0', 'N/A')}  |  Time (s12): {case.get('LongT5_Time_1', 'N/A')}  |  BERTScore F1 s7: {lt5_s7_score}  |  BERTScore F1 s12: {lt5_s12_score}")

    print(f"\n  MedGemma SUMMARY (Section 7):")
    print(wrap_output(case.get("MedGemma_Summary_0") or "N/A"))
    print(f"  MedGemma SUMMARY (Section 12):")
    print(wrap_output(case.get("MedGemma_Summary_1") or "N/A"))
    print(f"\n  ⏱ Time (s7): {case.get('MedGemma_Time_0', 'N/A')}  |  Time (s12): {case.get('MedGemma_Time_1', 'N/A')}  |  BERTScore F1 s7: {mg_s7_score}  |  BERTScore F1 s12: {mg_s12_score}")

    print(f"\n   Gemma4 SUMMARY (Section 7):")
    print(wrap_output(case.get("Gemma4_Summary_0") or "N/A"))
    print(f"  Gemma4 SUMMARY (Section 12):")
    print(wrap_output(case.get("Gemma4_Summary_1") or "N/A"))
    print(f"\n  ⏱ Time (s7): {case.get('Gemma4_Time_0', 'N/A')}  |  Time (s12): {case.get('Gemma4_Time_1', 'N/A')}  |  BERTScore F1 s7: {g4_s7_score}  |  BERTScore F1 s12: {g4_s12_score}")


  SHOWCASE — 3 EXAMPLE SUMMARIES

──────────────────────────────────────────────────────────────────────
  DRUG 31
  Source: https://dailymed.nlm.nih.gov/dailymed/fda/fdaDrugXsl.cfm?setid=fc22e8d
──────────────────────────────────────────────────────────────────────

  ORIGINAL TEXT (first 150 words):
  7 DRUG INTERACTIONS •Administration with rifampin or rifabutin is known to
  reduce atovaquone concentrations; concomitant use with MALARONE is not
  recommended. (7.1) •Proguanil may potentiate anticoagulant effect of warfarin
  and other coumarin-based anticoagulants. Caution advised when initiating or
  withdrawing MALARONE in patients on anticoagulants; coagulation tests should
  be closely monitored. (7.2) •Tetracycline may reduce atovaquone
  concentrations; parasitemia should be closely monitored. (7.3) 7.1
  Rifampin/Rifabutin Concomitant administration of rifampin or rifabutin is
  known to reduce atovaquone concentrations [see Clinical Pharmacology (12.3)].
  The concomitant a

#### Overall Comparison

In [ ]:
avg_lt5_score = round(sum(lt5_scores) / len(lt5_scores), 4)
avg_mg_score  = round(sum(mg_scores)  / len(mg_scores),  4)
avg_g4_score  = round(sum(g4_scores)  / len(g4_scores),  4)

avg_lt5_time  = round(sum(lt5_times)  / len(lt5_times),  2)
avg_mg_time   = round(sum(mg_times)   / len(mg_times),   2)
avg_g4_time   = round(sum(g4_times)   / len(g4_times),   2)

scores = {
    "Long-T5": avg_lt5_score,
    "MedGemma": avg_mg_score,
    "Gemma4": avg_g4_score
}

times = {
    "Long-T5": avg_lt5_time,
    "MedGemma": avg_mg_time,
    "Gemma4": avg_g4_time
}

winner = max(scores, key=scores.get)
faster = min(times, key=times.get)

print(f"\n{'='*75}")
print("  OVERALL RESULTS")
print(f"{'='*75}")
print(f"  {'Metric':<30} {'Long-T5':>10} {'MedGemma':>10} {'Gemma4':>10}")
print(f"  {'─'*70}")
print(f"  {'Avg BERTScore F1':<30} {avg_lt5_score:>10} {avg_mg_score:>10} {avg_g4_score:>10}")
print(f"  {'Avg Time per Summary (s)':<30} {avg_lt5_time:>10} {avg_mg_time:>10} {avg_g4_time:>10}")
print(f"  {'Total Drugs Evaluated':<30} {len(master_drug_data):>10}")
print(f"{'─'*75}")
print(f"  More Accurate Model : {winner} ({scores[winner]})")
print(f"  Faster Model        : {faster} ({times[faster]}s)")
print(f"{'='*75}")


  OVERALL RESULTS
  Metric                            Long-T5   MedGemma     Gemma4
  ──────────────────────────────────────────────────────────────────────
  Avg BERTScore F1                   0.8519      0.818     0.8027
  Avg Time per Summary (s)             4.33      17.13      10.39
  Total Drugs Evaluated                  72
───────────────────────────────────────────────────────────────────────────
  More Accurate Model : Long-T5 (0.8519)
  Faster Model        : Long-T5 (4.33s)
